In [ ]:
import pickle
import numpy as np
from lenstronomy.Workflow.fitting_sequence import FittingSequence
from lenstronomy.Util import param_util

class CustomUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if name == 'LikelihoodAddition':
            class LikelihoodAddition:
                pass
            return LikelihoodAddition
        return super().find_class(module, name)


import sys
import os
import contextlib

@contextlib.contextmanager
def suppress_stdout_stderr(to_file):
    """
    Redirect stdout and stderr to a file.
    """
    with open(to_file, 'w') as f:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = f
        sys.stderr = f
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

class LikelihoodAddition(object):
    """
    Gaussian prior enforcing alignment between lens mass PA
    and IR band lens light PA only.
    """

    def __init__(
        self,
        sigma_offset=10.0,          # degrees (1-sigma)
        has_perturber=False,
        lens_mass_index=0,
        perturber_mass_index=1,
        ir_light_indices=(0, 1),    # IR lens light components 
        perturber_ir_indices=None  # optionally have perturber mass-light alignment
    ):
        self.sigma_offset = sigma_offset
        self.has_perturber = has_perturber
        self.lens_mass_index = lens_mass_index
        self.perturber_mass_index = perturber_mass_index
        self.ir_light_indices = tuple(ir_light_indices)

        if perturber_ir_indices is None:
            self.perturber_ir_indices = ()
        else:
            self.perturber_ir_indices = tuple(perturber_ir_indices)

    def __call__(
        self,
        kwargs_lens=None,
        kwargs_source=None,
        kwargs_lens_light=None,
        kwargs_ps=None,
        kwargs_special=None,
        kwargs_extinction=None,
        kwargs_tracer_source=None,
    ):
        return self.logL_addition(kwargs_lens, kwargs_lens_light)

    # geometry helpers

    @staticmethod
    def _phi_deg(e1, e2):
        phi, _ = param_util.ellipticity2phi_q(e1, e2)
        return (phi * 180.0 / np.pi) % 180.0

    @staticmethod
    def _dphi(phi1, phi2):
        dphi = abs(phi1 - phi2)
        return dphi

    def _alignment_logL(self, phi_mass, kwargs_lens_light, indices):
        """
        Gaussian PA alignment prior for selected light components.
        """
        logL = 0.0

        for i in indices:
            if i >= len(kwargs_lens_light):
                continue

            comp = kwargs_lens_light[i]

            # Safety: skip UNIFORM components
            if 'e1' not in comp or 'e2' not in comp:
                continue

            phi_light = self._phi_deg(comp['e1'], comp['e2'])
            dphi = self._dphi(phi_mass, phi_light)

            logL += -0.5 * (dphi / self.sigma_offset) ** 2

        return logL

    def logL_addition(self, kwargs_lens, kwargs_lens_light):
        logL = 0.0

        # main lens
        phi_lens_mass = self._phi_deg(
            kwargs_lens[self.lens_mass_index]['e1'],
            kwargs_lens[self.lens_mass_index]['e2'],
        )

        logL += self._alignment_logL(
            phi_lens_mass,
            kwargs_lens_light,
            self.ir_light_indices,
        )

        # perturber logic
        if self.has_perturber and len(self.perturber_ir_indices) > 0:
            phi_pert_mass = self._phi_deg(
                kwargs_lens[self.perturber_mass_index]['e1'],
                kwargs_lens[self.perturber_mass_index]['e2'],
            )

            logL += self._alignment_logL(
                phi_pert_mass,
                kwargs_lens_light,
                self.perturber_ir_indices,
            )

        return logL

# Recreate logL object explicitly
if include_perturber == True:
    logL_PA = LikelihoodAddition(
        has_perturber=include_perturber,
        ir_light_indices=(0,),
        perturber_ir_indices=(1,)
    )
else:
    logL_PA = LikelihoodAddition(
        has_perturber=include_perturber,
        ir_light_indices=(0, 1)
    )

In [ ]:
filename = f"joint_modeling/{system_name}/{system_name}_joint.pkl"

with open(filename, "rb") as f:
    saved = CustomUnpickler(f).load()

fitting_seq_old = saved["fitting_seq"]
kwargs_model = saved["kwargs_model"]
kwargs_params = saved["kwargs_params"]
kwargs_constraints = saved["kwargs_constraints"]
kwargs_likelihood = saved["kwargs_likelihood"]
kwargs_likelihood['custom_logL_addition'] = logL_PA
kwargs_data_joint = saved["kwargs_data_joint"]
multi_band_list = saved["multi_band_list"]
chain_list = saved["chain_list"]

# Extract last walker positions (for flattened chain)

last_chain = chain_list[-1]
samples = last_chain[1]   # shape (total_samples, n_dim)

print("Samples shape:", samples.shape)

n_dim = samples.shape[1]
total_samples = samples.shape[0]

walker_ratio = 10  
n_walkers = walker_ratio * n_dim

# Safety check
if total_samples % n_walkers != 0:
    raise ValueError(
        f"Total samples ({total_samples}) not divisible by n_walkers ({n_walkers}). "
        "Check walkerRatio used in original run."
    )

n_steps = total_samples // n_walkers

print("n_dim:", n_dim)
print("n_walkers:", n_walkers)
print("n_steps:", n_steps)

# Reshape to (n_steps, n_walkers, n_dim)
chain = samples.reshape(n_steps, n_walkers, n_dim)

# Final walker positions
last_positions = chain[-1, :, :]   # shape (n_walkers, n_dim)

print("Recovered last walker positions:", last_positions.shape)

In [ ]:
fitting_seq = FittingSequence(
    kwargs_data_joint,
    kwargs_model,
    kwargs_constraints,
    kwargs_likelihood,
    kwargs_params
)

# continue MCMC
kwargs_mcmc_continue = {
    'n_burn': 0,  # no burn-in since we're continuing
    'n_run': n_additional_steps,
    'walkerRatio': 10,
    'init_samples': last_positions, # initial sample from where to start the MCMC process
    're_use_samples': True, # re-uses the samples described in init_samples
    'sigma_scale': 0.1
}

with suppress_stdout_stderr(f"joint_modeling/{system_name}/{system_name}_continue_mcmc_chain.txt"):
    chain_list_new = fitting_seq.fit_sequence([['MCMC', kwargs_mcmc_continue]])

chain_list.extend(chain_list_new)
kwargs_result = fitting_seq.best_fit()

# Save updated result
saved["chain_list"] = chain_list
saved["fitting_seq"] = fitting_seq
saved["kwargs_result"] = kwargs_result

with open(filename, "wb") as f:
    pickle.dump(saved, f)

print("MCMC successfully continued and saved.")